# Example 03 — Two-DOF Chain of Oscillators with Unilateral Spring

Frequency-response analysis of a 2-DOF chain with a unilateral (contact) spring at DOF 0, capturing the asymmetric impact nonlinearity via high-harmonic HB.

**System**: `mi=[1,1]`, `ki=[0,1,1]`, `di=0.03*ki`, unilateral spring k=100 gap=1 at DOF 0, force F=0.1 at DOF 1, H=21.

**Metric**: `a_rms = sqrt(sum(Q_dof0^2)) / sqrt(2)` — RMS of DOF 0 across all harmonics.

**Reference**: Krack & Gross 2019; MATLAB example `03_twoDOFoscillator_unilateralSpring`.

In [ ]:
# --- Imports ---
import sys
from pathlib import Path

import numpy as np
import scipy.linalg as _la
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

_repo_root = Path('..').resolve()
if str(_repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(_repo_root / 'src'))

from nlvib.nonlinearities.elements import unilateral_spring
from nlvib.systems.oscillators import ChainOfOscillators
from nlvib.solvers.harmonic_balance import hb_residual
from nlvib.continuation.solver import ContinuationSolver, ContinuationOptions
from nlvib.visualization.plots import plot_frf

In [ ]:
# --- System setup ---
# Matches MATLAB: mi=[1,1], ki=[0,1,1], di=0.03*ki
MASSES             = [1.0, 1.0]
STIFFNESSES        = [0.0, 1.0, 1.0]   # grounding stiffness=0, coupling=1, wall=1
DAMPINGS           = [0.0, 0.03, 0.03] # di = 0.03*ki
CONTACT_STIFFNESS  = 100.0
CONTACT_GAP        = 1.0
CONTACT_DOF        = 0                 # unilateral spring acts on DOF 0
EXCITATION_DOF     = 1                 # harmonic force applied at DOF 1
EXCITATION_AMP     = 0.1
H                  = 21                # number of harmonics (matches MATLAB H=21)
OMEGA_START        = 0.5
OMEGA_END          = 0.8
NEWTON_TOL         = 1e-8

system = ChainOfOscillators(masses=MASSES, stiffnesses=STIFFNESSES, dampings=DAMPINGS)
system.add_nonlinear_element(
    unilateral_spring(k=CONTACT_STIFFNESS, gap=CONTACT_GAP, dof_index=CONTACT_DOF)
)
print(f'System: {system}')
print(f'n_dof={system.n_dof}, H={H}, n_total={system.n_dof*(2*H+1)}')

In [ ]:
# --- Initial solution at OMEGA_END (linear frequency-domain guess) ---
# Trace from OMEGA_END=0.8 down to OMEGA_START=0.5, matching MATLAB Om_s=0.8->Om_e=0.5.
# Use negated-lambda trick so the continuation solver steps lambda from -0.8 toward -0.5.

n_dof   = system.n_dof
n_total = n_dof * (2 * H + 1)
excitation = {'dof': EXCITATION_DOF, 'amplitude': EXCITATION_AMP}

M_d = system.M.toarray()
D_d = system.D.toarray()
K_d = system.K.toarray()

Fex_c = np.zeros(n_dof, dtype=complex)
Fex_c[EXCITATION_DOF] = EXCITATION_AMP

Om_start_hb = OMEGA_END   # start continuation at 0.8
Q1_lin = _la.solve(-Om_start_hb**2 * M_d + 1j * Om_start_hb * D_d + K_d, Fex_c)
Q = np.zeros(n_total)
Q[(2*1-1)*n_dof : (2*1-1)*n_dof + n_dof] = Q1_lin.real
Q[2*1*n_dof     : 2*1*n_dof     + n_dof] = -Q1_lin.imag

R, J = hb_residual(Q, Om_start_hb, system, H, excitation)
print(f'Initial residual at omega={Om_start_hb:.3f}: {np.linalg.norm(R):.3e}')

In [ ]:
# --- Run HB continuation (negated-lambda: traces OMEGA_END -> OMEGA_START) ---
def neg_hb_residual(q, neg_om):
    return hb_residual(q, -neg_om, system, H, excitation)

opts = ContinuationOptions(
    verbose=False,
    ds_initial=0.02,
    ds_min=0.01,
    ds_max=0.06,
    max_steps=150,
    max_newton_iter=10,
    newton_tol=NEWTON_TOL,
    adapt_step=True,
    lambda_min=-(OMEGA_END + 0.05),
    lambda_max=-(OMEGA_START - 0.05),
)

result = ContinuationSolver().run(neg_hb_residual, Q, -Om_start_hb, opts)
print(f'Continuation: {result.message}')
print(f'Steps accepted: {result.n_steps}')

# Recover omega (negate lambda back)
solutions  = result.solutions
omega_branch = -solutions[:, -1]
Q_all      = solutions[:, :-1]
print(f'Omega range: [{omega_branch.min():.4f}, {omega_branch.max():.4f}]')

In [ ]:
# --- Extract a_rms (DOF 0, all harmonics) and save FRF plot ---
# a_rms = sqrt(sum(Q_dof0^2)) / sqrt(2)  — matches MATLAB metric
Q_dof0 = Q_all.reshape(Q_all.shape[0], 2*H+1, n_dof)[:, :, CONTACT_DOF]
a_rms  = np.sqrt(np.sum(Q_dof0**2, axis=1)) / np.sqrt(2)

mask = (omega_branch >= OMEGA_START) & (omega_branch <= OMEGA_END)

output_dir = Path('..') / 'examples' / '03_two_dof_unilateral' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

fig_frf, ax_frf = plt.subplots(figsize=(7, 4))
ax_frf.plot(omega_branch[mask], a_rms[mask], 'b-', linewidth=2)
ax_frf.set_xlabel('Excitation frequency (rad/s)')
ax_frf.set_ylabel('a_rms (DOF 0, all harmonics)')
ax_frf.set_title(f'Example 03 — Two-DOF Unilateral Spring: FRF (H={H})')
ax_frf.set_xlim(OMEGA_START, OMEGA_END)
ax_frf.grid(True, linestyle='--', alpha=0.5)
fig_frf.tight_layout()
fig_frf.savefig(output_dir / 'frf.png', dpi=150, bbox_inches='tight')
print('Saved frf.png')

# Peak amplitude / frequency
peak_idx   = int(np.argmax(a_rms[mask]))
omega_peak = float(omega_branch[mask][peak_idx])
amp_peak   = float(a_rms[mask][peak_idx])
print(f'Peak a_rms (DOF 0): {amp_peak:.6f} at omega={omega_peak:.4f} rad/s')

In [ ]:
# --- Display FRF inline ---
fig_frf

In [ ]:
# --- Summary ---
print('=' * 60)
print('SUMMARY — Example 03: Two-DOF Unilateral Spring')
print('=' * 60)
print(f'  System:  mi={MASSES}, ki={STIFFNESSES}, di={DAMPINGS}')
print(f'  Contact: k={CONTACT_STIFFNESS}, gap={CONTACT_GAP}, DOF={CONTACT_DOF}')
print(f'  Force:   F={EXCITATION_AMP} at DOF {EXCITATION_DOF}')
print(f'  H={H}, omega in [{OMEGA_START}, {OMEGA_END}] rad/s')
print(f'')
print(f'  Peak a_rms (DOF {CONTACT_DOF}, all H): {amp_peak:.6f}')
print(f'  Frequency at peak:                     {omega_peak:.4f} rad/s')
print(f'  Total continuation steps:              {result.n_steps}')
print(f'  Omega range covered:                   [{omega_branch.min():.4f}, {omega_branch.max():.4f}]')
print('=' * 60)
# Validate against comparison notebook MATLAB-validated values
# MATLAB reference: peak a_rms ~ 0.9809, omega_peak ~ 0.7541
MATLAB_PEAK_ARMS  = 0.980930
MATLAB_PEAK_OMEGA = 0.7541
err_amp   = abs(amp_peak   - MATLAB_PEAK_ARMS)  / MATLAB_PEAK_ARMS
err_omega = abs(omega_peak - MATLAB_PEAK_OMEGA) / MATLAB_PEAK_OMEGA
print(f'  vs MATLAB: peak a_rms err={err_amp*100:.2f}%, peak omega err={err_omega*100:.2f}%')
assert err_amp < 0.01,   f'Peak amplitude error {err_amp:.4f} > 1%'
assert err_omega < 0.01, f'Peak frequency error {err_omega:.4f} > 1%'
print('  PASS: both errors < 1%')